# 可穿戴数据按周期提取

从 5 个源表中仅保留与 `cycle_anchor_clean` 中完整周期对应的数据，各自输出到 `subdataset/` 下的新文件。

- **heart_rate**, **heart_rate_variability_details**, **wrist_temperature**, **resting_heart_rate**：按 `(id, study_interval, day_in_study)` 在锚点集合内筛选。
- **computed_temperature**：按 `(id, study_interval, sleep_end_day_in_study)` 在锚点集合内筛选，并增加列 `day_in_study = sleep_end_day_in_study`。

In [1]:
import pandas as pd
from pathlib import Path

# 路径：notebook 在 data_process/ 下时 ROOT = main_workspace；否则尝试 main_workspace 子目录
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "data_process" else cwd
if not (ROOT / "subdataset" / "cycle_anchor_clean.csv").exists() and (cwd / "main_workspace").exists():
    ROOT = cwd / "main_workspace"
CYCLE_ANCHOR_PATH = ROOT / "subdataset" / "cycle_anchor_clean.csv"
MCPHASES_DIR = ROOT / "mcphases-a-dataset-of-physiological-hormonal-and-self-reported-events-and-symptoms-for-menstrual-health-tracking-with-wearables-1.0.0"
OUT_DIR = ROOT / "subdataset"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("CYCLE_ANCHOR_PATH:", CYCLE_ANCHOR_PATH)
print("MCPHASES_DIR exists:", MCPHASES_DIR.exists())
print("OUT_DIR:", OUT_DIR)

CYCLE_ANCHOR_PATH: /Users/xujing/FYP/main_workspace/subdataset/cycle_anchor_clean.csv
MCPHASES_DIR exists: True
OUT_DIR: /Users/xujing/FYP/main_workspace/subdataset


In [2]:
# 加载周期锚点，构建 (id, study_interval, day_in_study) 集合
cycle = pd.read_csv(CYCLE_ANCHOR_PATH)
cycle_keys = set(zip(cycle["id"], cycle["study_interval"], cycle["day_in_study"]))
print("cycle_anchor_clean 行数:", len(cycle))
print("唯一 (id, study_interval, day_in_study) 数:", len(cycle_keys))

cycle_anchor_clean 行数: 2797
唯一 (id, study_interval, day_in_study) 数: 2797


In [3]:
# 1. resting_heart_rate：日级，直接按 (id, study_interval, day_in_study) 内连接
cycle_days = cycle[["id", "study_interval", "day_in_study"]].drop_duplicates()
rhr = pd.read_csv(MCPHASES_DIR / "resting_heart_rate.csv")
rhr_cycle = rhr.merge(cycle_days, on=["id", "study_interval", "day_in_study"], how="inner")
rhr_cycle.to_csv(OUT_DIR / "resting_heart_rate_cycle.csv", index=False)
print("resting_heart_rate: 原始", len(rhr), "行 -> 周期内", len(rhr_cycle), "行")

resting_heart_rate: 原始 13737 行 -> 周期内 6736 行


In [4]:
# 2. heart_rate_variability_details：按 (id, study_interval, day_in_study) 内连接
hrv = pd.read_csv(MCPHASES_DIR / "heart_rate_variability_details.csv")
hrv_cycle = hrv.merge(cycle_days, on=["id", "study_interval", "day_in_study"], how="inner")
hrv_cycle.to_csv(OUT_DIR / "heart_rate_variability_details_cycle.csv", index=False)
print("heart_rate_variability_details: 原始", len(hrv), "行 -> 周期内", len(hrv_cycle), "行")

heart_rate_variability_details: 原始 436262 行 -> 周期内 224246 行


In [5]:
# 3. computed_temperature：按 sleep_end_day_in_study 归属日与周期对齐，保留整行并增加 day_in_study 列
ct = pd.read_csv(MCPHASES_DIR / "computed_temperature.csv")
ct_cycle = ct.merge(
    cycle_days.rename(columns={"day_in_study": "sleep_end_day_in_study"}),
    on=["id", "study_interval", "sleep_end_day_in_study"],
    how="inner",
)
# 便于后续与日级表对齐
ct_cycle["day_in_study"] = ct_cycle["sleep_end_day_in_study"]
ct_cycle.to_csv(OUT_DIR / "computed_temperature_cycle.csv", index=False)
print("computed_temperature: 原始", len(ct), "行 -> 周期内", len(ct_cycle), "行")

computed_temperature: 原始 5575 行 -> 周期内 2768 行


In [6]:
# 4. wrist_temperature：按 (id, study_interval, day_in_study) 筛选，分块读入以控制内存
CHUNK_SIZE = 500_000
wt_path = MCPHASES_DIR / "wrist_temperature.csv"
out_wt = OUT_DIR / "wrist_temperature_cycle.csv"
first = True
total_in, total_out = 0, 0
for chunk in pd.read_csv(wt_path, chunksize=CHUNK_SIZE):
    total_in += len(chunk)
    c = chunk.merge(cycle_days, on=["id", "study_interval", "day_in_study"], how="inner")
    total_out += len(c)
    c.to_csv(out_wt, mode="w" if first else "a", header=first, index=False)
    first = False
print("wrist_temperature: 原始", total_in, "行 -> 周期内", total_out, "行 ->", out_wt.name)

wrist_temperature: 原始 6856019 行 -> 周期内 3601898 行 -> wrist_temperature_cycle.csv


In [7]:
# 5. heart_rate：按 (id, study_interval, day_in_study) 筛选，分块读入（约 6300 万行）
CHUNK_SIZE_HR = 2_000_000
hr_path = MCPHASES_DIR / "heart_rate.csv"
out_hr = OUT_DIR / "heart_rate_cycle.csv"
first = True
total_in, total_out = 0, 0
for chunk in pd.read_csv(hr_path, chunksize=CHUNK_SIZE_HR):
    total_in += len(chunk)
    c = chunk.merge(cycle_days, on=["id", "study_interval", "day_in_study"], how="inner")
    total_out += len(c)
    c.to_csv(out_hr, mode="w" if first else "a", header=first, index=False)
    first = False
print("heart_rate: 原始", total_in, "行 -> 周期内", total_out, "行 ->", out_hr.name)

heart_rate: 原始 63100276 行 -> 周期内 32619798 行 -> heart_rate_cycle.csv


In [8]:
# 输出文件概览
for name in [
    "resting_heart_rate_cycle.csv",
    "heart_rate_variability_details_cycle.csv",
    "computed_temperature_cycle.csv",
    "wrist_temperature_cycle.csv",
    "heart_rate_cycle.csv",
]:
    p = OUT_DIR / name
    if p.exists():
        print(p.name, "->", f"{p.stat().st_size / 1024 / 1024:.2f} MB")
    else:
        print(p.name, "-> 未生成")

resting_heart_rate_cycle.csv -> 0.33 MB
heart_rate_variability_details_cycle.csv -> 11.80 MB
computed_temperature_cycle.csv -> 0.37 MB
wrist_temperature_cycle.csv -> 155.41 MB
heart_rate_cycle.csv -> 965.35 MB


## 各周期表空值 / 0 值 / 空字符串检查

统计每个 `*_cycle.csv` 中：**NaN**、**空字符串**、**0** 的行数及占比（0 与空视为“空值”）。

In [ ]:
def report_null_zero_empty(df, table_name):
    """统计 DataFrame 中 NaN、0、空字符串的行数与比例，并输出行级「至少一处空值」的统计。"""
    n = len(df)
    print(f"\n{'='*60}\n【{table_name}】 总行数: {n}\n{'='*60}")
    if n == 0:
        return
    # 各列 NaN 数
    nan_cnt = df.isna().sum()
    # 各列 0 数（仅数值列）
    zero_cnt = pd.Series(0, index=df.columns)
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            zero_cnt[c] = (df[c] == 0).sum()
    # 各列空字符串数（对象/字符串列）
    empty_cnt = pd.Series(0, index=df.columns)
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype) == "string":
            empty_cnt[c] = (df[c].astype(str).str.strip() == "").sum()

    print("\n--- 按列：NaN / 0 / 空字符串 行数 (比例) ---")
    for c in df.columns:
        nn, zn, en = nan_cnt[c], zero_cnt[c], empty_cnt[c]
        nr, zr, er = nn / n * 100, zn / n * 100, en / n * 100
        parts = [f"NaN:{nn}({nr:.2f}%)"]
        if zn > 0:
            parts.append(f"0:{zn}({zr:.2f}%)")
        if en > 0:
            parts.append(f"空串:{en}({er:.2f}%)")
        print(f"  {c}: {', '.join(parts)}")

    # 行级：至少有一个 NaN / 0 / 空字符串 的行数
    rows_any_nan = df.isna().any(axis=1).sum()
    mask_zero = pd.Series(False, index=df.index)
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            mask_zero |= (df[c] == 0) & df[c].notna()
    rows_any_empty = pd.Series(False, index=df.index)
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype) == "string":
            rows_any_empty |= (df[c].astype(str).str.strip() == "")
    rows_any_bad = (df.isna().any(axis=1) | mask_zero | rows_any_empty).sum()
    print(f"\n行级汇总: 至少一处 NaN 的行数 = {rows_any_nan} ({rows_any_nan/n*100:.2f}%)")
    print(f"          至少一处 0 的行数   = {mask_zero.sum()} ({mask_zero.sum()/n*100:.2f}%)")
    print(f"          至少一处空串的行数 = {rows_any_empty.sum()} ({rows_any_empty.sum()/n*100:.2f}%)")
    print(f"          至少一处(NaN/0/空串)的行数 = {rows_any_bad} ({rows_any_bad/n*100:.2f}%)")

In [ ]:
def report_null_zero_empty_chunked(path, table_name, chunksize=1_000_000):
    """大表分块读入，汇总各列 NaN/0/空串 行数及总行数，并估算「至少一处空值」行数。"""
    total_rows = 0
    nan_sum = None
    zero_sum = None
    empty_sum = None
    rows_any_nan = 0
    rows_any_zero = 0
    rows_any_empty = 0
    for chunk in pd.read_csv(path, chunksize=chunksize):
        n = len(chunk)
        total_rows += n
        if nan_sum is None:
            nan_sum = chunk.isna().sum()
            zero_sum = pd.Series(0.0, index=chunk.columns)
            for c in chunk.columns:
                if pd.api.types.is_numeric_dtype(chunk[c]):
                    zero_sum[c] = (chunk[c] == 0).sum()
            empty_sum = pd.Series(0, index=chunk.columns)
            for c in chunk.columns:
                if chunk[c].dtype == object or str(chunk[c].dtype) == "string":
                    empty_sum[c] = (chunk[c].astype(str).str.strip() == "").sum()
        else:
            nan_sum += chunk.isna().sum()
            for c in chunk.columns:
                if pd.api.types.is_numeric_dtype(chunk[c]):
                    zero_sum[c] += (chunk[c] == 0).sum()
                if chunk[c].dtype == object or str(chunk[c].dtype) == "string":
                    empty_sum[c] += (chunk[c].astype(str).str.strip() == "").sum()
        rows_any_nan += chunk.isna().any(axis=1).sum()
        mask_z = pd.Series(False, index=chunk.index)
        for c in chunk.columns:
            if pd.api.types.is_numeric_dtype(chunk[c]):
                mask_z |= (chunk[c] == 0) & chunk[c].notna()
        rows_any_zero += mask_z.sum()
        mask_e = pd.Series(False, index=chunk.index)
        for c in chunk.columns:
            if chunk[c].dtype == object or str(chunk[c].dtype) == "string":
                mask_e |= (chunk[c].astype(str).str.strip() == "")
        rows_any_empty += mask_e.sum()

    print(f"\n{'='*60}\n【{table_name}】(分块统计) 总行数: {total_rows}\n{'='*60}")
    if total_rows == 0:
        return
    print("\n--- 按列：NaN / 0 / 空字符串 行数 (比例) ---")
    for c in nan_sum.index:
        nn, zn = nan_sum[c], zero_sum[c]
        en = empty_sum[c] if c in empty_sum else 0
        nr, zr, er = nn / total_rows * 100, zn / total_rows * 100, en / total_rows * 100
        parts = [f"NaN:{int(nn)}({nr:.2f}%)"]
        if zn > 0:
            parts.append(f"0:{int(zn)}({zr:.2f}%)")
        if en > 0:
            parts.append(f"空串:{int(en)}({er:.2f}%)")
        print(f"  {c}: {', '.join(parts)}")
    print(f"\n行级汇总: 至少一处 NaN 的行数 = {rows_any_nan} ({rows_any_nan/total_rows*100:.2f}%)")
    print(f"          至少一处 0 的行数   = {rows_any_zero} ({rows_any_zero/total_rows*100:.2f}%)")
    print(f"          至少一处空串的行数 = {rows_any_empty} ({rows_any_empty/total_rows*100:.2f}%)")

In [ ]:
# 对 5 个周期表依次做空值检查；大表（>200MB）分块统计
CYCLE_FILES = [
    "resting_heart_rate_cycle.csv",
    "heart_rate_variability_details_cycle.csv",
    "computed_temperature_cycle.csv",
    "wrist_temperature_cycle.csv",
    "heart_rate_cycle.csv",
]
THRESHOLD_MB = 200
for fname in CYCLE_FILES:
    path = OUT_DIR / fname
    if not path.exists():
        print(f"\n跳过（文件不存在）: {fname}")
        continue
    size_mb = path.stat().st_size / 1024 / 1024
    name = fname.replace("_cycle.csv", "")
    if size_mb > THRESHOLD_MB:
        report_null_zero_empty_chunked(path, name, chunksize=500_000)
    else:
        df = pd.read_csv(path)
        report_null_zero_empty(df, name)